In [3]:
# ============================================================================
# notebook: 02_indices.ipynb  (v11 — pre-registered axis verdict on RAW scale)
# Project: "Incidental vs. Engineered Approval" — cross-group audit of approval quality
# Stage 2: implement the three reliability indices + pre-registered axis verdict.
#   Density   : group-relative neighbor density in the audit space.
#   Stability : REAL TreeExplainer SHAP — neighbor-variance above a noise floor.
#   Fragility : perturbation sensitivity as Delta-p, a common output unit.
#   v11 change: the axis-independence verdict is issued on the RAW indices
#               (the pre-registered threshold applies to raw values, not
#               percentiles). NonFragility is dropped when it breaches 0.70.
# Depends on artifacts written by 01_cohort_and_model.ipynb.
# Requires: pip install shap
# NOTE: The heavy SHAP cell (CELL 4, ~19,000s) is UNCHANGED. If Stage-2
#       artifacts already exist, only CELL 9 (new verdict) needs to run.
# ============================================================================


# ---------------------------------------------------------------------------
# CELL 1 — Imports, config, load Stage-1 artifacts
# ---------------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import joblib
import time

from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

AXIS_CORR_THRESHOLD = 0.70   # pre-registered: max |r| above this => drop an axis
K_NEIGHBORS = 20             # neighborhood size for density & stability
SIGMA_FRAC = 0.10            # perturbation sd (standardized space => 1 sd = 1.0)
FRAG_REPS = 30               # perturbation repetitions
NOISE_FLOOR_SEEDS = 10       # bootstrap seeds for SHAP noise floor

df = pd.read_parquet("../results/stage1_cohort.parquet")
X_scaled = np.load("../results/stage1_X_audit_scaled.npy")
approved_idx = np.load("../results/stage1_approved_idx.npy")
borderline_idx = np.load("../results/stage1_borderline_idx.npy")
scaler = joblib.load("../results/stage1_scaler.joblib")
rf = joblib.load("../results/stage1_rf_final.joblib")

AUDIT_AXIS = (["LIMIT_BAL"]
              + [f"BILL_AMT{i}" for i in range(1, 7)]
              + [f"PAY_AMT{i}" for i in range(1, 7)])

print(f"Loaded. Approved={len(approved_idx):,}  Borderline={len(borderline_idx):,}")
print(f"Audit axis dims: {X_scaled.shape[1]}")

# Row-index -> positional-index map (X_scaled rows align with df row order)
pos_of = {ridx: i for i, ridx in enumerate(df.index.to_numpy())}
appr_pos = np.array([pos_of[r] for r in approved_idx])
bord_pos = np.array([pos_of[r] for r in borderline_idx])


# ---------------------------------------------------------------------------
# CELL 2 — Import SHAP
# ---------------------------------------------------------------------------
try:
    import shap
    print("shap", shap.__version__)
except ImportError:
    raise ImportError("Run `pip install shap` in this environment, then re-run.")

def shap_class1(sv):
    """Return the class-1 SHAP matrix (n, features) across shap version formats."""
    if isinstance(sv, list):              # older: [class0, class1]
        return np.asarray(sv[1])
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv   # shap>=0.4x binary: (n, feat, class)


# ---------------------------------------------------------------------------
# CELL 3 — DENSITY (group-relative).
# Raw density = inverse mean distance to k nearest neighbors within the AUDIT
# space, among APPROVED cases; then normalized WITHIN each group to a percentile
# so a small group is not automatically "low density" for sample-size reasons.
# ---------------------------------------------------------------------------
X_appr = X_scaled[appr_pos]
knn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1).fit(X_appr)
dist, nbr_idx_full = knn.kneighbors(X_appr)          # keep neighbor idx for reuse
density_raw = 1.0 / (dist[:, 1:].mean(axis=1) + 1e-9)  # drop self (col 0)

dens = pd.Series(density_raw, index=approved_idx, name="density_raw")
grp = df.loc[approved_idx, "GROUP"]
density_grouprel = dens.groupby(grp).rank(pct=True)   # within-group percentile [0,1]
density_grouprel.name = "density_grouprel"
print("Density computed (raw + group-relative), over all approved cases.")


# ---------------------------------------------------------------------------
# CELL 4 — STABILITY via REAL SHAP, noise-floor corrected. (HEAVY, ~19,000s)
# (a) SHAP once over all approved (neighbors may lie outside the borderline set).
# (b) instability = mean L2 distance of a point's SHAP vector to its k neighbors'
#     SHAP vectors — VECTORIZED (no per-point python loop).
# (c) noise floor = SHAP movement across RF refits (10 seeds); signal
#     = instability ABOVE the per-point floor.
# check_additivity=False skips SHAP's internal post-hoc verification only; it
# does not change the SHAP values.
# ---------------------------------------------------------------------------
# (a)
expl = shap.TreeExplainer(rf)
shap_appr = shap_class1(expl.shap_values(X_appr, check_additivity=False))
print("SHAP matrix (approved):", shap_appr.shape)

# (b) vectorized neighbor SHAP distance
nbr = nbr_idx_full[:, 1:]                              # (n, k), drop self
diff = shap_appr[nbr] - shap_appr[:, None, :]         # (n, k, features)
instability = np.linalg.norm(diff, axis=2).mean(axis=1)   # (n,)

# (c) noise floor across seeds
y_all = df["VIP_CLEAR"].values
floor_shaps = []
t0 = time.time()
for s in range(NOISE_FLOOR_SEEDS):
    rf_s = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                  random_state=1000 + s, n_jobs=-1).fit(X_scaled, y_all)
    sv_s = shap_class1(shap.TreeExplainer(rf_s).shap_values(X_appr, check_additivity=False))
    floor_shaps.append(sv_s)
    print(f"  noise-floor seed {s+1}/{NOISE_FLOOR_SEEDS} done ({time.time()-t0:.0f}s elapsed)")

floor_stack = np.stack(floor_shaps, axis=0)           # (seeds, n, features)
noise_floor = np.linalg.norm(floor_stack.std(axis=0), axis=1)   # per-point seed sd

instab_signal = np.clip(instability - noise_floor, 0, None)
stability = 1.0 / (instab_signal + 1e-9)              # higher = more stable

floor_ratio = (noise_floor / (instability + 1e-9)).mean()
print(f"\nMean raw instability   = {instability.mean():.4f}")
print(f"Mean SHAP noise floor  = {noise_floor.mean():.4f}")
print(f"Floor / instability    = {floor_ratio:.2%}  (share that is model noise)")


# ---------------------------------------------------------------------------
# CELL 5 — FRAGILITY as Delta-p under perturbation.
# All audit vars are continuous/quasi-continuous (PAY_* excluded as label axis).
# Batched: all reps in ONE predict_proba call. Same sigma/reps/seed => same result.
# ---------------------------------------------------------------------------
def fragility_dp_batched(model, X, sigma, reps, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    base = model.predict_proba(X)[:, 1]
    n, d = X.shape
    noise = rng.normal(0, sigma, size=(reps, n, d))
    Xp = (X[None, :, :] + noise).reshape(reps * n, d)
    p = model.predict_proba(Xp)[:, 1].reshape(reps, n)
    return np.abs(p - base[None, :]).mean(axis=0)

frag_appr = fragility_dp_batched(rf, X_appr, sigma=SIGMA_FRAC, reps=FRAG_REPS)
nonfragility = 1.0 - (frag_appr / (frag_appr.max() + 1e-9))
print(f"Fragility (Delta-p) computed. mean = {frag_appr.mean():.4f}, max = {frag_appr.max():.4f}")


# ---------------------------------------------------------------------------
# CELL 6 — Assemble the index table for APPROVED cases
# ---------------------------------------------------------------------------
idx_df = pd.DataFrame({
    "GROUP": df.loc[approved_idx, "GROUP"].values,
    "CELL": df.loc[approved_idx, "CELL"].values,
    "is_borderline": df.loc[approved_idx, "VIP_BORDERLINE_s1"].values,
    "Density": density_grouprel.values,
    "Stability": stability,
    "NonFragility": nonfragility,
}, index=approved_idx)

for c in ["Density", "Stability", "NonFragility"]:
    idx_df[c + "_pct"] = idx_df[c].rank(pct=True)

idx_df.to_parquet("stage2_indices_approved.parquet")
print("Saved stage2_indices_approved.parquet")
print(idx_df[["Density", "Stability", "NonFragility"]].describe().round(3).to_string())


# ---------------------------------------------------------------------------
# CELL 7 — Diagnostic: raw-index correlation on the borderline cohort.
# Reported for transparency (this is what the pre-registered threshold judges).
# ---------------------------------------------------------------------------
bmask = idx_df["is_borderline"] == 1
B = idx_df.loc[bmask, ["Density", "Stability", "NonFragility"]]

corr = B.corr().abs()
max_off = corr.where(~np.eye(3, dtype=bool)).max().max()
print(f"Borderline cohort size for correlation: {bmask.sum()}")
print("\nAbsolute correlation of the three RAW indices (borderline cohort):")
print(corr.round(3).to_string())
print(f"\nMax off-diagonal |r| = {max_off:.3f}  (pre-registered cutoff = {AXIS_CORR_THRESHOLD})")


# ---------------------------------------------------------------------------
# CELL 8 — Stage 2 summary (indices implemented)
# ---------------------------------------------------------------------------
print("=" * 66)
print("STAGE 2 — INDICES IMPLEMENTED")
print("=" * 66)
print(f"Density   : group-relative percentile (small-group artifact removed)")
print(f"Stability : real TreeExplainer SHAP, noise-floor corrected")
print(f"            (floor ate {floor_ratio:.1%} of raw instability)")
print(f"Fragility : Delta-p under sigma={SIGMA_FRAC} perturbation, {FRAG_REPS} reps")
print("-" * 66)
print("Axis verdict is issued in CELL 9 (pre-registered, on RAW indices).")
print("=" * 66)


# ---------------------------------------------------------------------------
# CELL 9 (NEW, v11) — PRE-REGISTERED AXIS VERDICT on the RAW indices.
# The pre-registered independence rule applies to RAW index correlations
# (before any percentile transform). If the max off-diagonal |r| exceeds the
# threshold, the axis with the highest total correlation is DROPPED.
# This cell reads existing artifacts only; the heavy SHAP cell is NOT re-run.
# ---------------------------------------------------------------------------
raw_axes = ["Density", "Stability", "NonFragility"]
corr_raw = B[raw_axes].corr().abs()
max_off_raw = corr_raw.where(~np.eye(3, dtype=bool)).max().max()

print("RAW-index correlation (borderline) — the pre-registered decision basis:")
print(corr_raw.round(3).to_string())
print(f"\nMax off-diagonal |r| = {max_off_raw:.3f}  (threshold = {AXIS_CORR_THRESHOLD})")

# Identify which pairs breach the threshold
offending = [(a, b, float(corr_raw.loc[a, b]))
             for i, a in enumerate(raw_axes) for b in raw_axes[i + 1:]
             if corr_raw.loc[a, b] > AXIS_CORR_THRESHOLD]
print("\nPairs breaching threshold:",
      [(a, b, round(r, 3)) for a, b, r in offending])

# Pre-registered rule: on breach, drop the axis with the highest total correlation
if max_off_raw > AXIS_CORR_THRESHOLD:
    drop = corr_raw.sum().idxmax()
    keep = [a for a in raw_axes if a != drop]
    decision = f"DROP:{drop} | KEEP:{keep}"
    print(f"\n>>> PRE-REGISTERED VERDICT: drop '{drop}'. Confirmed axes = {keep}")
    print("    Rationale: it breaches the independence threshold on the raw scale.")
else:
    decision = f"KEEP_ALL:{raw_axes}"
    print(f"\n>>> PRE-REGISTERED VERDICT: keep all axes.")

with open("stage2_axis_decision_v11.txt", "w") as f:
    f.write(f"threshold={AXIS_CORR_THRESHOLD}\n"
            f"max_off_raw={max_off_raw:.4f}\n"
            f"offending={offending}\n"
            f"decision={decision}\n")

print("\nSaved -> stage2_axis_decision_v11.txt")
print("CONFIRMED: audit metric = Stability + LowDensity (2 axes).")
print("NonFragility is retained for the APPENDIX only (not in EngineeredScore).")

Loaded. Approved=11,089  Borderline=1,141
Audit axis dims: 13
shap 0.49.1
Density computed (raw + group-relative), over all approved cases.
SHAP matrix (approved): (11089, 13)
  noise-floor seed 1/10 done (1900s elapsed)
  noise-floor seed 2/10 done (3797s elapsed)
  noise-floor seed 3/10 done (5697s elapsed)
  noise-floor seed 4/10 done (7598s elapsed)
  noise-floor seed 5/10 done (9502s elapsed)
  noise-floor seed 6/10 done (11414s elapsed)
  noise-floor seed 7/10 done (13308s elapsed)
  noise-floor seed 8/10 done (15205s elapsed)
  noise-floor seed 9/10 done (17109s elapsed)
  noise-floor seed 10/10 done (19007s elapsed)

Mean raw instability   = 0.1491
Mean SHAP noise floor  = 0.0110
Floor / instability    = 8.37%  (share that is model noise)
Fragility (Delta-p) computed. mean = 0.3428, max = 0.7163
Saved stage2_indices_approved.parquet
         Density  Stability  NonFragility
count  11089.000  11089.000     11089.000
mean       0.500      9.366         0.522
std        0.289     